# 2. Run as web service (Integration)

As we saw in [Lesson1](01_get_started.ipynb), the clinet in Foundry Local directly calls Foundry core and ONNX runtime core library. Because of this manner, it cannot be used with other development tools, and developers should always use the Foundry SDK directly.

By exposing Foundry Local endpoint as OpenAI compliant web service, you can integrate with a wide variety tools - such as, OpenAI SDK, LangChain, etc.

## Load model

Same as [Lesson1](01_get_started.ipynb), let's load a model to start.

In [1]:
from foundry_local_sdk import Configuration, FoundryLocalManager

# initialize manager and execution providers (see Lesson1)
config = Configuration(app_name="foundry_local_samples")
FoundryLocalManager.initialize(config)
manager = FoundryLocalManager.instance
manager.download_and_register_eps()

# load model (see Lesson1)
model = manager.catalog.get_model("qwen2.5-0.5b")
model.download()
model.load()

## Start web service

Let's expose endpoint as a local web service, and get URL of endpoint.<br>
As I have mentioned above, this is OpenAI compatible endpoint.

In [2]:
# start web service
manager.start_web_service()

# get endpoint url
base_url = f"{manager.urls[0]}/v1"
print(base_url)

http://127.0.0.1:52971/v1


## Invoke with OpenAI SDK

Now let's invoke this endpoint by using OpenAI SDK as follows.

> Note : For using LangChain with Foundry Local, see [official how-to document](https://learn.microsoft.com/en-us/azure/foundry-local/how-to/how-to-use-langchain-with-foundry-local).

In [3]:
import openai

client = openai.OpenAI(
    base_url=base_url,
    api_key="none",
)

response = client.chat.completions.create(
    model=model.id,
    messages=[
        {"role": "system", "content": "You are a helpful assistant, named Foundry AI."},
        {"role": "user", "content": "Hello, Foundry AI"}
    ],
)

print(response.choices[0].message.content)

Hello! It seems like you might be trying to contact Foundry AI. Could you please provide me with some more details? What specific information would you like to ask or discuss about this platform or project? I'm here to help if I can, and will do my best to assist you in any way I can.


## Stop web service

After it's done, you can stop the running web service as follows.

In [4]:
manager.stop_web_service()